<a href="https://colab.research.google.com/github/Hion-cy/ClassFiles/blob/main/RDD_Al263158.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Materia:** Ingeniería de Datos Avanzada
###**Alumna:** Carmen Yolanda Hion Vela
###**Matrícula:** AL263158

#Actividad práctica: RDD en PySpark

##Planteamiento

Genera una lista de datos numéricos utilizando una distribución normal aleatoria en Python. Posteriormente, convierte estos datos en un RDD y realiza diferentes operaciones para analizarlos

### Librerias

In [8]:
import random
import math
from pyspark.sql import SparkSession




##Instrucciones

###Genera una lista de al menos 1000 valores usando una función de Python.

Usa una distribución normal con parámetros controlados para que los resultados sean interpretables:

* Media (μ): 50

* Desviación estándar (σ): 10

* Tamaño de muestra: 1000 valores

* Fijar semilla a 18



In [26]:
print("\n" + "-" * 80)
print("SECCIÓN A: TRANSFORMACIONES BÁSICAS")
print("-" * 80)

#Crear lista de valores
random.seed(18)
mu, sigma, n= 50,10,1000
lista_datos = [random.gauss(mu, sigma) for _ in range(n)]



--------------------------------------------------------------------------------
SECCIÓN A: TRANSFORMACIONES BÁSICAS
--------------------------------------------------------------------------------


###Paraleliza la lista para crear un RDD llamado rdd0


In [27]:
#Crear sesion de Spark
print("\n" + "-" * 80)
print("CREANDO SPARKSESSION")
print("-" * 80)

spark= SparkSession.builder.appName("RDD").getOrCreate()

#Crear sparkContext
sc = spark.sparkContext

print(f"\n✓ SparkContext creado exitosamente")
print(f"  - Nombre de la aplicación: {sc.appName}")
print(f"  - Master: {sc.master}")
print(f"  - Versión de Spark: {sc.version}")
print(f"  - Python Version: {sc.pythonVer}")
print(f"  - Default Parallelism: {sc.defaultParallelism}")

#Lista paralelizada
rdd = sc.parallelize(lista_datos)
#print(data)


--------------------------------------------------------------------------------
CREANDO SPARKSESSION
--------------------------------------------------------------------------------

✓ SparkContext creado exitosamente
  - Nombre de la aplicación: RDD
  - Master: local[*]
  - Versión de Spark: 4.0.2
  - Python Version: 3.12
  - Default Parallelism: 2


####Comentarios adicionales


En esta etapa del ejercicio se inicializó el entorno de Spark y se utiliza parallelize para distribuir los elementos de la lista aleatoria y convertirla en un rdd distribuido

### Aplica las siguientes operaciones:

####Transformaciones

Usa map() para elevar al cuadrado cada elemento.

Usa take(5) para mostrar 5 resultados

Usa filter() para quedarte solo con valores que sean mayores a 50. Imprime en pantalla el resultado de 4 resultados.

In [24]:
#Take de los 5 originales
rdd_original = rdd.take(5)
print(f"RDD original: {rdd_original}")

# Map para elevar elementos al cuadrado
rdd_cuadrado = rdd.map(lambda x: x**2)

#Mostrar 5 elementos
print(f"RDD cuadrado: {rdd_cuadrado.take(5)}")

RDD original: [56.16038215368986, 63.36618488548809, 46.63484796432865, 55.72625752659761, 38.3523841014258]
RDD cuadrado: [3153.9885236484865, 4015.273386941859, 2174.809044656048, 3105.4157779206766, 1470.9053662632982]


In [25]:
#Filtro para valores mayores de 50
rdd_filtrado = rdd.filter(lambda x: x > 50)

print(f"RDD filtrado: {rdd_filtrado.take(4)}")

RDD filtrado: [56.16038215368986, 63.36618488548809, 55.72625752659761, 50.76842269939075]


####Comentarios adicionales
Las transformaciones facilitan el trabajo con los datos a pesar de trabajar con una lista de 1000 elementos (Y no revisarlos individualmente). El uso de filter ademas, permite una limpieza mas rapida basada en criterios especificos


### Transformaciones en Pseudo-Conjuntos

#### Genera dos conjuntos de datos (listas) de 1000 valores cada uno utilizando una distribución normal con:

* Media (μ) = 50

* Desviación estándar (σ) = 10

In [71]:
# --- SECCIÓN B: TRANSFORMACIONES EN PSEUDO-CONJUNTOS ---
print("\n" + "-" * 80)
print("SECCIÓN B: OPERACIONES DE CONJUNTOS")
print("-" * 80)

mu, sigma, n= 50,10,1000
# Se realizaran las listas con numeros enteros para incrementar la
# probabilidad de obtener numeros duplicados para el ejercicio de transformaciones
random.seed(18)
lista1= [int(random.gauss(mu, sigma)) for _ in range(n)]
#random.seed(18)
lista2= [int(random.gauss(mu, sigma)) for _ in range(n)]


--------------------------------------------------------------------------------
SECCIÓN B: OPERACIONES DE CONJUNTOS
--------------------------------------------------------------------------------


####Convierte ambas listas en RDDs:

* rdd1

* rdd2

In [72]:
#Paralelizar las listas
rdd1=sc.parallelize(lista1)
rdd2=sc.parallelize(lista2)


####Aplica las siguientes transformaciones de pseudo-conjuntos:

* Usa union() para combinar ambos RDDs.

* Usa distinct() para eliminar valores duplicados del resultado anterior.

* Usa intersection() para obtener los valores comunes entre rdd1 y rdd2.

* Usa subtract() para obtener los valores que están en rdd1 pero no en rdd2.

* Usa cartesian() para generar todas las posibles combinaciones entre ambos RDDs.
###Ejecuta acciones para observar los resultados:

* Usa take() o takeOrdered() para visualizar 5 elementos de cada transformación.

* Usa count() para conocer el tamaño de cada RDD resultante.


In [92]:
# Combinar rdd
rdd_union_rdd = rdd1.union(rdd2)
total_registros=rdd_union_rdd.count()
print(f"Total de registros: {total_registros}")
# Eliminar duplicados
rdd_unicos = rdd_union_rdd.distinct()
total_unicos = rdd_unicos.count()
print(f"Total de registros únicos: {total_unicos}")
print("Ejemplo de registros únicos:")
for val in rdd_unicos.takeOrdered(5, key=None):
    print(val)

cantidad_duplicados = total_registros - total_unicos
print(f"Cantidad de duplicados eliminados: {cantidad_duplicados}")

Total de registros: 2000
Total de registros únicos: 63
Ejemplo de registros únicos:
11
19
20
21
22
Cantidad de duplicados eliminados: 1937


In [91]:
#Valores comunes
rdd_interseccion = rdd1.intersection(rdd2)
print(f"Total de valores comunes: {rdd_interseccion.count()}")
print("Valores comunes:")
for val in rdd_interseccion.takeOrdered(5, key=None):
    print(val)

Total de valores comunes: 55
Valores comunes:
21
24
25
26
27


In [95]:
print(f"Tamaño de RDD1: {rdd1.count()}")
print(f"Tamaño de RDD2: {rdd2.count()}")
print(f"Tamaño despues de limpiar: {rdd_unicos.count()}")
#Elementos de rdd1 que no estan en rdd2
rdd1_unicos=rdd1.subtract(rdd2)
rdd2_unicos=rdd2.subtract(rdd1)

#print("Todos los valores unicos")
#for val in rdd_unicos.collect():
#    print(val)

print("Valores únicos de rdd1:")
for val in rdd1_unicos.take(5):
    print(val)


Tamaño de RDD1: 1000
Tamaño de RDD2: 1000
Tamaño despues de limpiar: 63
Valores únicos de rdd1:
20
81
81
19
11


In [85]:
# Combinaciones posibles
rdd_cartesian = rdd1.cartesian(rdd2)
print(f"Total de combinaciones posibles: {rdd_cartesian.count()}")
print("Ejemplo de combinaciones posibles:")
for val in rdd_cartesian.take(5):
    print(val)

Total de combinaciones posibles: 1000000
Ejemplo de combinaciones posibles:
(56, 51)
(56, 61)
(56, 36)
(56, 53)
(56, 59)


####Comentarios adicionales

En esta sección se utilizaron valores aleatorios de tipo entero para aumentar la probabilidad de encontrar coincidencias entre ambos RDD:

El objetivo del cambio es validar visualmente cómo las operaciones gestionan la duplicidad de los datos con operaciones como *intersection* o *subtract*. Mientras que *cartesian* permite generar combinaciones en grandes volúmenes lo que facilita el cruce de un elemento del Rdd1 con todos los de otro cualquiera para encontrar patrones.

###Operaciones matemáticas y estadísticas.
* Utiliza rdd0 y calcula el promedio, valor máximo y mínimo.
* Calcula la varianza y desviación estándar
* Utiliza la biblioteca math y calcula la raíz cuadrada. Imprime solo 5 resultados.

In [105]:

# Informacion del rdd0
print("\n" + "=" * 80)
print("ESTADÍSTICAS")
print("=" * 80)

media= rdd.mean()
maximo= rdd.max()
minimo= rdd.min()
varianza= rdd.variance()
desviacion_estandar= rdd.stdev()
print(f"Media: {media:.2f}")
print(f"Máximo: {maximo}")
print(f"Mínimo: {minimo}")
print(f"Varianza: {varianza:.2f}")
print(f"Desviación estándar: {desviacion_estandar:.2f}")

print("Raiz cuadrada de RDD0")
raiz= rdd.map(lambda x: math.sqrt(abs(x)))
for val in raiz.take(5):
    print(val)



ESTADÍSTICAS
Media: 50.28
Máximo: 81.57038277154388
Mínimo: 11.987484747459995
Varianza: 102.30
Desviación estándar: 10.11
Raiz cuadrada de RDD0
7.494023095353381
7.9602879901099115
6.828971222982906
7.465002178606354
6.192930170882423


####Comentarios adicionales
El uso de acciones estadi+ísticas permite ampliar el contexto de los rdd, proporcionando metricas para el analisis de la distribucion de los datos.

##Conclusión


En esta práctica se implementa el manejo de datos a gran escala. Utilizando el modelo de programación de los RDD es posible observar que el volumen de los datos no deberia de ser un impedimento para realizar un análisis concreto apoyandose de las librerias adecuadas.
Ademas de que Spark proporciona una sintaxis bastante sencilla y con un amplio espectro de operaciones que se pueden utilizar.